In [1]:
import ibis
import os
import duckdb
import fiona
import geopandas as gpd

ibis.options.interactive = True

In [94]:
# Path to the GeoPackage file with multiple layers of geospatial data
geopackage_path = "input/gadm41_IND.gpkg"

In [96]:
# List all layer names in the GeoPackage using Fiona
layer_names = fiona.listlayers(geopackage_path)
layer_names

['ADM_ADM_0', 'ADM_ADM_1', 'ADM_ADM_2', 'ADM_ADM_3']

In [47]:
# Connect to DB
con = ibis.duckdb.connect('db/india_model.ddb', extensions = ['spatial'])

In [76]:
# Iterate over each layer in the GeoPackage
for layer_name in layer_names:
    # Read the current layer into a GeoDataFrame and convert geometries to WKB format
    gdf = gpd.read_file(geopackage_path, layer=layer_name).to_wkb()

    # Create an Ibis in-memory table from the GeoDataFrame data
    ibis_memtable = ibis.memtable(gdf)
    
    # Convert WKB geometry column to DuckDB geometry type for spatial operations
    ibis_memtable = ibis_memtable.mutate(geometry=ibis_memtable.geometry.cast("geometry"))

    # Create a table in DuckDB for the current layer, overwriting if it already exists
    con.create_table(layer_name, ibis_memtable, overwrite=True)

    # Clean up the in-memory table to free memory
    del ibis_memtable

In [97]:
con.list_tables()

['ADM_ADM_0',
 'ADM_ADM_1',
 'ADM_ADM_2',
 'ADM_ADM_3',
 'ibis_pandas_memtable_wy7zr4glwrb3bgx6guiqktemay']

In [74]:
con.table('ADM_ADM_2')

┏━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ GID_2     ┃ GID_0  ┃ COUNTRY ┃ GID_1   ┃ NAME_1              ┃ NL_NAME_1 ┃ NAME_2                   ┃ VARNAME_2            ┃ NL_NAME_2 ┃ TYPE_2   ┃ ENGTYPE_2 ┃ CC_2   ┃ HASC_2   ┃ geometry                                                                         ┃
┡━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ string    │ string │ string  │ string  │ string              │ string    │ string                   │ string               │ string    │ string   │ string    │ string │ string   │ geospatial:geometry                                                              │
├───────────┼────────┼─────────┼─────────┼─────────────────────┼───────────┼──────────────────────────┼──────────────────────┼───────────┼──────────┼───────────┼────────┼──────────┼──────────────────────────────────────────────────────────────────────────────────┤
│ IND.1.1_1 │ IND    │ India   │ IND.1_1 │ Andaman and Nicobar │ NA        │ Nicobar Islands          │ NA                   │ NA        │ District │ District  │ NA     │ IN.AN.NI │ <MULTIPOLYGON (((93.79 6.852, 93.79 6.852, 93.791 6.852, 93.791 6.851, 93.79...> │
│ IND.1.2_1 │ IND    │ India   │ IND.1_1 │ Andaman and Nicobar │ NA        │ North and Middle Andaman │ NA                   │ NA        │ District │ District  │ NA     │ IN.AN.NM │ <MULTIPOLYGON (((92.844 12.15, 92.845 12.15, 92.845 12.15, 92.845 12.15, 92....> │
│ IND.1.3_1 │ IND    │ India   │ IND.1_1 │ Andaman and Nicobar │ NA        │ South Andaman            │ NA                   │ NA        │ District │ District  │ NA     │ IN.AN.SA │ <MULTIPOLYGON (((92.521 10.897, 92.523 10.895, 92.523 10.895, 92.526 10.891,...> │
│ IND.2.1_1 │ IND    │ India   │ IND.2_1 │ Andhra Pradesh      │ NA        │ Anantapur                │ Anantpur, Ananthapur │ NA        │ District │ District  │ NA     │ IN.AD.AN │ <MULTIPOLYGON (((77.846 13.928, 77.83 13.927, 77.822 13.931, 77.818 13.935, ...> │
│ IND.2.2_1 │ IND    │ India   │ IND.2_1 │ Andhra Pradesh      │ NA        │ Chittoor                 │ Chitoor|Chittor      │ NA        │ District │ District  │ NA     │ IN.AD.CH │ <MULTIPOLYGON (((78.546 12.744, 78.55 12.739, 78.546 12.729, 78.551 12.726, ...> │
│ IND.2.3_1 │ IND    │ India   │ IND.2_1 │ Andhra Pradesh      │ NA        │ East Godavari            │ NA                   │ NA        │ District │ District  │ NA     │ IN.AD.EG │ <MULTIPOLYGON (((82.317 16.574, 82.317 16.574, 82.317 16.574, 82.317 16.573,...> │
│ IND.2.4_1 │ IND    │ India   │ IND.2_1 │ Andhra Pradesh      │ NA        │ Guntur                   │ NA                   │ NA        │ District │ District  │ NA     │ IN.AD.GU │ <MULTIPOLYGON (((80.783 15.837, 80.783 15.837, 80.784 15.837, 80.784 15.836,...> │
│ IND.2.5_1 │ IND    │ India   │ IND.2_1 │ Andhra Pradesh      │ NA        │ Krishna                  │ Kistna               │ NA        │ District │ District  │ NA     │ IN.AD.KR │ <MULTIPOLYGON (((81.03 15.765, 81.03 15.765, 81.03 15.765, 81.03 15.765, 81....> │
│ IND.2.6_1 │ IND    │ India   │ IND.2_1 │ Andhra Pradesh      │ NA        │ Kurnool                  │ NA                   │ NA        │ District │ District  │ NA     │ IN.AD.KU │ <MULTIPOLYGON (((77.897 15.17, 77.888 15.17, 77.882 15.169, 77.874 15.136, 7...> │
│ IND.2.7_1 │ IND    │ India   │ IND.2_1 │ Andhra Pradesh      │ NA        │ Nellore                  │ NA                   │ NA        │ District │ District  │ NA     │ IN.AD.NE │ <MULTIPOLYGON (((80.194 13.52, 80.194 13.52, 80.194 13.52, 80.194 13.519, 80...> │
│ …         │ …      │ … 

In [79]:
import lonboard

In [91]:
lonboard.viz(
    con.table('ADM_ADM_2'), 
    polygon_kwargs={
        "get_fill_color": [255, 0, 0, 50],      # Semi-transparent red fill color
        "get_line_color": [0, 0, 0, 255],       # Black boundary lines
        "stroked": True,                        # Draw polygon boundaries
        "pickable": True,                       # Allow picking for interactivity
        "line_width_min_pixels": 1,             # Minimum width of the boundary lines
        "extruded": False,                      # Keep the layer 2D (no 3D extrusion)
        "auto_highlight": True,                 # Enable automatic highlighting on hover
    })

Map(basemap_style=<CartoBasemap.DarkMatter: 'https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json'…